# EDA — Walmart Sales

Quick exploratory analysis backing the modeling choices in `ARCHITECTURE.md`
(gradient boosting, calendar + lag/rolling features, holiday-week rule,
economic indicators) and useful for the presentation's technical deep dive.

Run `python scripts/setup_data.py` first so `data/raw/Walmart_Sales.csv` exists.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from demand_forecast.data.ingest import load_walmart_sales

RAW_DIR = Path.cwd().parent / "data" / "raw"
df = load_walmart_sales(RAW_DIR)
print(df.shape, df["store_nbr"].nunique(), "stores")
df.head()

## Total sales over time across all 45 stores — holiday-week spikes stand out

In [ ]:
weekly_total = df.groupby("date")["sales"].sum()
fig, ax = plt.subplots(figsize=(12, 4))
weekly_total.plot(ax=ax, label="Total weekly sales")
holiday_dates = df.loc[df["holiday_flag"] == 1, "date"].unique()
for d in holiday_dates:
    ax.axvline(d, color="red", alpha=0.3, linestyle="--")
ax.set_title("Total weekly sales across all stores (red = holiday week)")
ax.set_ylabel("Total sales (USD)")
plt.show()

## Store scale varies ~8x — motivates the store-size fairness segment

In [ ]:
avg_by_store = df.groupby("store_nbr")["sales"].mean().sort_values()
avg_by_store.plot(kind="bar", figsize=(14, 4), title="Average weekly sales by store")
plt.ylabel("Avg weekly sales (USD)")
plt.show()
print(f"Smallest store avg: ${avg_by_store.min():,.0f} | Largest: ${avg_by_store.max():,.0f}")

## Holiday weeks sell more on average — validates `holiday_flag` as a feature

In [ ]:
by_holiday = df.groupby("holiday_flag")["sales"].agg(["mean", "count"])
by_holiday.index = ["Non-holiday week", "Holiday week"]
by_holiday

## Economic indicators (CPI, unemployment) vary meaningfully by store — the regional-conditions proxy used in `docs/RESPONSIBLE_AI.md`

In [ ]:
econ_by_store = df.groupby("store_nbr")[["cpi", "unemployment"]].mean()
econ_by_store.describe()

## Sales vs. temperature and fuel price (weak, non-linear relationships — motivates a tree-based model over linear regression)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(df["temperature"], df["sales"], s=5, alpha=0.3)
axes[0].set_xlabel("Temperature (F)")
axes[0].set_ylabel("Weekly sales")
axes[1].scatter(df["fuel_price"], df["sales"], s=5, alpha=0.3)
axes[1].set_xlabel("Fuel price (USD/gal)")
plt.tight_layout()
plt.show()